In [ ]:
!pip install -q transformers sentencepiece accelerate faiss-cpu sentence-transformers ragas datasets rouge-score bert-score sacrebleu matplotlib pandas tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━

In [ ]:
import os
import gc
import json
import torch
import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from sentence_transformers import SentenceTransformer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/MALAYALAM_RAG_PROJECT"

FAISS_DIR = f"{BASE_DIR}/faiss_index"
EMBED_DIR = f"{BASE_DIR}/embeddings"

SARVAM_DIR = f"{BASE_DIR}/sarvam_rag"
BLOOMZ_DIR = f"{BASE_DIR}/bloomz_rag"
MT5_DIR = f"{BASE_DIR}/mt5_rag"
MBART_DIR = f"{BASE_DIR}/mbart_rag"

for d in [
    BASE_DIR,
    FAISS_DIR,
    EMBED_DIR,
    SARVAM_DIR,
    BLOOMZ_DIR,
    MT5_DIR,
    MBART_DIR
]:
    os.makedirs(d, exist_ok=True)

In [ ]:
train_df = pd.read_csv("/content/drive/MyDrive/train.csv")
valid_df = pd.read_csv("/content/drive/MyDrive/validation.csv")
test_df = pd.read_csv("/content/drive/MyDrive/test.csv")

train_df = train_df[["article", "summary"]].dropna()
valid_df = valid_df[["article", "summary"]].dropna()
test_df = test_df[["article", "summary"]].dropna()

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

(7758, 2)
(970, 2)
(970, 2)


In [ ]:
def chunk_text(text, chunk_size=1000, overlap=150):
    chunks = []

    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

In [ ]:
all_chunks = []

for article in tqdm(train_df["article"]):
    chunks = chunk_text(str(article))
    all_chunks.extend(chunks)

chunk_df = pd.DataFrame({"chunk": all_chunks})

chunk_df.to_csv(
    f"{BASE_DIR}/chunk_corpus.csv",
    index=False
)

print(len(chunk_df))

100%|██████████| 7758/7758 [00:00<00:00, 750482.27it/s]

7809


In [ ]:
embed_model = SentenceTransformer(
    "intfloat/multilingual-e5-base",
    device="cuda"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [ ]:
BATCH_SIZE = 64

all_embeddings = []

chunks = chunk_df["chunk"].tolist()

for i in tqdm(range(0, len(chunks), BATCH_SIZE)):
    batch = chunks[i:i+BATCH_SIZE]

    emb = embed_model.encode(
        batch,
        convert_to_numpy=True,
        show_progress_bar=False,
        normalize_embeddings=True
    )

    all_embeddings.append(emb)

embeddings = np.vstack(all_embeddings)

np.save(
    f"{EMBED_DIR}/chunk_embeddings.npy",
    embeddings
)

print(embeddings.shape)

100%|██████████| 123/123 [00:54<00:00,  2.26it/s]

(7809, 768)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

faiss.write_index(
    index,
    f"{FAISS_DIR}/malayalam_faiss.index"
)

In [ ]:
def retrieve_context(query, top_k=2):
    q_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(q_emb, top_k)

    retrieved = [
        chunk_df.iloc[idx]["chunk"]
        for idx in indices[0]
    ]

    return retrieved

In [ ]:
sample_article = valid_df.iloc[0]["article"]

contexts = retrieve_context(sample_article)

for i, c in enumerate(contexts):
    print(f"\nCONTEXT {i+1}")
    print(c[:500])


CONTEXT 1
കണ്ണൂർ: തുടർച്ചയായി എത്തുന്ന മഴയും മണ്ണിടിച്ചിലും കാരണം കണ്ണൂർ വിമാനത്താവളത്തിന് പരിസരത്ത് താമസിക്കുന്ന കുടുംബങ്ങൾ ഭീഷണിയിലായിരിക്കുകയാണ്.

CONTEXT 2
കൽപറ്റ: മഴക്കെടുതിയിൽ വീട് നഷ്ടപ്പെട്ട് ആട്ടിൻ കൂട്ടിൽ അഭയം തേടിയിരിക്കുകയാണ് വയനാട് നടവയലിലെ ഒരുകുടുംബം.


In [ ]:
pd.DataFrame({
    "query": [sample_article],
    "retrieved_1": [contexts[0]],
    "retrieved_2": [contexts[1]]
}).to_csv(
    f"{BASE_DIR}/retrieval_samples.csv",
    index=False
)

In [ ]:
!pip install -q ragas rouge-score bert-score sacrebleu

In [ ]:
import numpy as np
import sacrebleu
import json
import matplotlib.pyplot as plt

from rouge_score import rouge_scorer
from bert_score import score

In [ ]:
def compute_basic_metrics(predictions, references):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=False
    )

    r1, r2, rl = [], [], []

    for ref, pred in zip(references, predictions):
        scores = scorer.score(ref, pred)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)

    rouge1 = np.mean(r1)
    rouge2 = np.mean(r2)
    rougeL = np.mean(rl)

    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references]
    ).score

    P, R, F1 = score(
        predictions,
        references,
        lang="ml",
        verbose=True
    )

    bertscore = F1.mean().item()

    return {
        "rouge1": rouge1,
        "rouge2": rouge2,
        "rougeL": rougeL,
        "bleu": bleu,
        "bertscore": bertscore
    }

In [ ]:
!pip install -q ragas

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision, context_recall

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_2267/263144184.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_precision, context_recall
/tmp/ipykernel_2267/263144184.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import

In [ ]:
#HS = 1 - (0.4F + 0.2CP + 0.2CR + 0.2B)

In [ ]:
def compute_hallucination_scores(
    faithfulness_score,
    cp,
    cr,
    bertscore,
    threshold=0.35
):
    hs = 1 - (
        0.4 * faithfulness_score +
        0.2 * cp +
        0.2 * cr +
        0.2 * bertscore
    )

    hr = 1 if hs > threshold else 0

    return hs, hr

In [ ]:
def save_metrics_plot(metrics, save_path):
    names = list(metrics.keys())
    values = list(metrics.values())

    plt.figure(figsize=(10, 6))
    plt.bar(names, values)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

SARVAM_MODEL = "sarvamai/sarvam-2b-v0.5"

sarvam_tokenizer = AutoTokenizer.from_pretrained(
    SARVAM_MODEL
)

sarvam_model = AutoModelForCausalLM.from_pretrained(
    SARVAM_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

sarvam_model.eval()

config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.70M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(64128, 2048)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
   

In [ ]:
def build_sarvam_rag_prompt(article, contexts):
    context_text = "\n".join(contexts)

    prompt = f"""
നിങ്ങൾ ഒരു മലയാളം വാർത്താ സംഗ്രഹ നിർമാതാവാണ്.

താഴെ നൽകിയിരിക്കുന്ന പശ്ചാത്തല വിവരങ്ങളും ലേഖനവും ഉപയോഗിച്ച്
2-3 വാചകങ്ങളിലായി ചെറിയ മലയാളം സംഗ്രഹം മാത്രം സൃഷ്ടിക്കുക.

പശ്ചാത്തല വിവരം:
{context_text}

ലേഖനം:
{article}

മലയാളം സംഗ്രഹം മാത്രം:
"""
    return prompt

In [ ]:
def generate_sarvam_summary(article):
    contexts = retrieve_context(article, top_k=3)

    prompt = build_sarvam_rag_prompt(article, contexts)

    inputs = sarvam_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():
        output = sarvam_model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            num_beams=2,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    pred = sarvam_tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    pred = pred.replace(prompt, "").strip()

    return pred, contexts

In [ ]:
RUN_ROWS = 200

In [ ]:
RUN_ROWS = len(valid_df)

In [ ]:
articles = []
references = []
predictions = []
retrieved_contexts = []

subset = valid_df.iloc[:RUN_ROWS]

for _, row in tqdm(subset.iterrows(), total=len(subset)):
    article = str(row["article"])
    reference = str(row["summary"])

    pred, ctx = generate_sarvam_summary(article)

    articles.append(article)
    references.append(reference)
    predictions.append(pred)
    retrieved_contexts.append(ctx)

100%|██████████| 970/970 [35:30<00:00,  2.20s/it]


In [ ]:
sarvam_pred_df = pd.DataFrame({
    "article": articles,
    "reference": references,
    "prediction": predictions,
    "contexts": retrieved_contexts
})

sarvam_pred_df.to_csv(
    f"{SARVAM_DIR}/predictions.csv",
    index=False
)

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

In [ ]:
sim_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    device="cuda"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def compute_basic_metrics(predictions, references):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=False
    )

    r1, r2, rl = [], [], []

    for ref, pred in zip(references, predictions):
        scores = scorer.score(ref, pred)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)

    rouge1 = np.mean(r1)
    rouge2 = np.mean(r2)
    rougeL = np.mean(rl)

    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references]
    ).score

    ref_emb = sim_model.encode(
        references,
        convert_to_tensor=True,
        batch_size=16
    )

    pred_emb = sim_model.encode(
        predictions,
        convert_to_tensor=True,
        batch_size=16
    )

    sims = util.cos_sim(pred_emb, ref_emb).diagonal()
    bertscore = sims.mean().item()

    return {
        "rouge1": rouge1,
        "rouge2": rouge2,
        "rougeL": rougeL,
        "bleu": bleu,
        "bertscore": bertscore
    }

In [ ]:
basic_metrics = compute_basic_metrics(
    predictions,
    references
)

print(basic_metrics)

{'rouge1': np.float64(0.007628865979381443), 'rouge2': np.float64(0.0002577319587628866), 'rougeL': np.float64(0.007628865979381443), 'bleu': 0.4923698359686007, 'bertscore': 0.5195360779762268}


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

ragas_embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base"
)

print("Embeddings ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings ready


In [ ]:
import pandas as pd

pred_df = pd.read_csv(f"{BASE_DIR}/sarvam_rag/predictions.csv")

articles = pred_df["article"].tolist()
references = pred_df["reference"].tolist()
predictions = pred_df["prediction"].tolist()

print(len(pred_df))
pred_df.head()

970


,article,reference,prediction,contexts
0,വയനാട്ടിൽ കാരാപ്പുഴ റിസർവോയർ പദ്ധതിക്കായി കുടി...,"കാരാപ്പുഴ പദ്ധതി: പകരം ഭൂമി നൽകിയില്ല, കുടിയിറ...",കണ്ഠൂര്: തുടര്ച്ചയായ മഴയെത്തുടര്ന്ന് കണ്ണൂര് വ...,['കണ്ണൂർ: തുടർച്ചയായി എത്തുന്ന മഴയും മണ്ണിടിച്...
1,കാബൂള്: തനിക്ക് പരിക്കൊന്നുമില്ലെന്നും പരിക്കി...,"എനിക്ക് പരിക്കില്ല, ടീമില് നിന്ന് ഒഴിവാക്കിയതാ...",അഫ്ഗാനിസ്ഥാന് പര്യടനത്തില് നിന്ന് വിരാട് കോഹ്ല...,['തിരുവല്ല: കാന്സര് ഇല്ലാത്ത രോഗിക്ക് കീമോതെറാ...
2,തിരുവനന്തപുരം: കേരള സർക്കാർ കായിക യുവജന കാര്യാ...,സ്കൂൾ കുട്ടികൾക്കായി ബാസ്ക്കറ്റ് ബോൾ പരിശീലന പ...,കേരള സർക്കാർ സ്പോൺസർ ചെയ്യുന്ന അടിസ്ഥാന തല ബാസ...,['സ്കൂൾതലം മുതൽ ബാസ്ക്കറ്റ്ബോളിൽ അന്താരാഷ്ട്ര ...
3,തിരുവനന്തപുരം: രാജ്യത്ത് ഗോഡ്സെ ക്ഷേത്രങ്ങള് വ...,ഗോ ഡ്സെ യ്ക്ക് ക്ഷേ ത്ര ങ്ങ ൾ നി ർ മ്മി ക്കു ...,കേരളത്തിൽ തീവ്രവാദ ബന്ധം ആരോപിച്ച് സിപിഎമ്മിന്...,['തിരുവനന്തപുരം: തീവ്രവാദ ബന്ധമുള്ള സിപിഎം പ്ര...
4,തിരുവനന്തപുരം : കാട്ടാക്കട പാറശ്ശാല നിയോജകമണ്ഡ...,മുടങ്ങിക്കിടന്ന മുഴുവന് പൊതുമരാമത്ത് പ്രവര്ത്ത...,കേരളത്തിലെ പൊതുഗതാഗത സംവിധാനത്തിന് ഗുണകരമായ നട...,['ആലപ്പുഴ: നിർമാണത്തിൽ കൃത്രിമം കാണിക്കാത്ത കര...


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [ ]:
from bert_score import BERTScorer
import evaluate
import sacrebleu
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

def compute_basic_metrics_v2(predictions, references):

    predictions = [str(x) for x in predictions]
    references = [str(x) for x in references]

    rouge = evaluate.load("rouge")

    rouge_scores = rouge.compute(
        predictions=predictions,
        references=references
    )

    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references]
    ).score

    scorer = BERTScorer(
        model_type="bert-base-multilingual-cased",
        device="cuda"
    )

    P, R, F1 = scorer.score(predictions, references)

    metrics = {
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"],
        "bleu": bleu,
        "bertscore": F1.mean().item()
    }

    return metrics, F1.numpy()

In [ ]:
pred_df = pd.read_csv(f"{BASE_DIR}/sarvam_rag/predictions.csv")

pred_df["article"] = pred_df["article"].fillna("").astype(str)
pred_df["reference"] = pred_df["reference"].fillna("").astype(str)
pred_df["prediction"] = pred_df["prediction"].fillna("").astype(str)

articles = pred_df["article"].tolist()
references = pred_df["reference"].tolist()
predictions = pred_df["prediction"].tolist()

print("Total:", len(predictions))
print("Empty predictions:", sum([p.strip()=="" for p in predictions]))

Total: 970
Empty predictions: 6


In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_allocated()/1024**3, "GB allocated")

8.766812324523926 GB allocated


In [ ]:
import evaluate
import sacrebleu
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

def compute_basic_metrics_clean(predictions, references):

    predictions = [str(x) for x in predictions]
    references = [str(x) for x in references]

    # ROUGE
    rouge = evaluate.load("rouge")
    rouge_scores = rouge.compute(
        predictions=predictions,
        references=references
    )

    # BLEU
    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references]
    ).score

    # "BERTScore-like" semantic similarity
    sim_model = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        device="cpu"
    )

    pred_emb = sim_model.encode(
        predictions,
        batch_size=16,
        show_progress_bar=True
    )

    ref_emb = sim_model.encode(
        references,
        batch_size=16,
        show_progress_bar=True
    )

    bert_like_scores = [
        cosine_similarity(
            pred_emb[i].reshape(1, -1),
            ref_emb[i].reshape(1, -1)
        )[0][0]
        for i in range(len(predictions))
    ]

    return {
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"],
        "bleu": bleu,
        "bertscore": float(np.mean(bert_like_scores))
    }, np.array(bert_like_scores)

In [ ]:
basic_metrics, bert_per_sample = compute_basic_metrics_clean(
    predictions,
    references
)

print(basic_metrics)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/61 [00:00<?, ?it/s]

Batches:   0%|          | 0/61 [00:00<?, ?it/s]

{'rouge1': np.float64(0.007560137457044674), 'rouge2': np.float64(0.0002577319587628866), 'rougeL': np.float64(0.007560137457044674), 'bleu': 0.4923698359686007, 'bertscore': 0.519536018371582}


In [ ]:
import json

with open(f"{BASE_DIR}/basic_metrics.json", "w") as f:
    json.dump(basic_metrics, f, indent=4)

In [ ]:
from sentence_transformers import util
from tqdm import tqdm
import numpy as np

def compute_faithfulness(articles, predictions):
    faith_scores = []

    for article, pred in tqdm(zip(articles, predictions), total=len(articles)):
        contexts = retrieve_context(article)

        combined_context = " ".join(contexts)

        emb_pred = embedding_model.encode(
            pred,
            convert_to_tensor=True
        )

        emb_ctx = embedding_model.encode(
            combined_context,
            convert_to_tensor=True
        )

        sim = util.cos_sim(emb_pred, emb_ctx).item()
        faith_scores.append(sim)

    return np.array(faith_scores)

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-base",
    device="cpu"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
faithfulness_scores = compute_faithfulness(
    articles,
    predictions
)

print(np.mean(faithfulness_scores))

100%|██████████| 970/970 [07:45<00:00,  2.08it/s]

0.8530455552425581


In [ ]:
def compute_context_precision(articles, predictions):
    cp_scores = []

    for article, pred in tqdm(zip(articles, predictions), total=len(articles)):
        contexts = retrieve_context(article)
        combined_context = " ".join(contexts)

        pred_sentences = pred.split(".")

        if len(pred_sentences) == 0:
            cp_scores.append(0)
            continue

        supported = 0

        emb_ctx = embedding_model.encode(
            combined_context,
            convert_to_tensor=True
        )

        for sent in pred_sentences:
            if len(sent.strip()) < 5:
                continue

            emb_sent = embedding_model.encode(
                sent,
                convert_to_tensor=True
            )

            sim = util.cos_sim(emb_sent, emb_ctx).item()

            if sim > 0.65:
                supported += 1

        cp = supported / max(len(pred_sentences), 1)
        cp_scores.append(cp)

    return np.array(cp_scores)

In [ ]:
cp_scores = compute_context_precision(
    articles,
    predictions
)

print(np.mean(cp_scores))

 99%|█████████▉| 961/970 [08:30<00:04,  1.82it/s]

In [ ]:
def compute_context_recall(articles, predictions):
    cr_scores = []

    for article, pred in tqdm(zip(articles, predictions), total=len(articles)):
        contexts = retrieve_context(article)

        covered = 0

        emb_pred = embedding_model.encode(
            pred,
            convert_to_tensor=True
        )

        for ctx in contexts:
            emb_ctx = embedding_model.encode(
                ctx,
                convert_to_tensor=True
            )

            sim = util.cos_sim(emb_pred, emb_ctx).item()

            if sim > 0.65:
                covered += 1

        cr = covered / max(len(contexts), 1)
        cr_scores.append(cr)

    return np.array(cr_scores)

In [ ]:
cr_scores = compute_context_recall(
    articles,
    predictions
)

print(np.mean(cr_scores))

In [ ]:
bert_norm = bert_per_sample

hallucination_scores = 1 - (
    0.4 * faithfulness_scores +
    0.2 * cp_scores +
    0.2 * cr_scores +
    0.2 * bert_norm
)

hallucination_binary = hallucination_scores > 0.4

hallucination_rate = hallucination_binary.mean()

print("Hallucination Rate:", hallucination_rate)
print("Mean HS:", hallucination_scores.mean())

In [ ]:
final_df = pd.DataFrame({
    "article": articles,
    "reference": references,
    "prediction": predictions,
    "bertscore": bert_norm,
    "faithfulness": faithfulness_scores,
    "context_precision": cp_scores,
    "context_recall": cr_scores,
    "hallucination_score": hallucination_scores,
    "hallucinated": hallucination_binary
})

final_df.to_csv(
    f"{BASE_DIR}/sarvam_rag_full_results.csv",
    index=False
)

In [ ]:
summary = {
    **basic_metrics,
    "faithfulness": float(np.mean(faithfulness_scores)),
    "context_precision": float(np.mean(cp_scores)),
    "context_recall": float(np.mean(cr_scores)),
    "hallucination_score": float(np.mean(hallucination_scores)),
    "hallucination_rate": float(hallucination_rate)
}

with open(f"{BASE_DIR}/sarvam_rag_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

print(summary)